# 07 — Embeddings & Semantic Search

## Learning requirements
Trước RAG, phải build retrieval không có generation.

Pipeline:

```text
Documents -> Split -> Chunks -> Embeddings -> Vector Store
                                          |
Query -> Query embedding -----------------+-> similarity -> top-k chunks
```

Hiểu:
- chunk size/overlap;
- metadata;
- similarity;
- top-k;
- recall vs precision;
- embedding model consistency.

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs = [
    Document(
        page_content="Refund requests are accepted within 14 days if the service has not been consumed.",
        metadata={"source": "refund-policy.md", "section": "refund"},
    ),
    Document(
        page_content="Enterprise customers can request SSO configuration from the admin portal.",
        metadata={"source": "enterprise.md", "section": "security"},
    ),
    Document(
        page_content="Tickets are confirmed only after successful payment.",
        metadata={"source": "ticketing.md", "section": "checkout"},
    ),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=180, chunk_overlap=20)
chunks = splitter.split_documents(docs)
for c in chunks:
    print(c.metadata, "=>", c.page_content)

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
while not (root / "requirements.txt").exists() and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

from src.providers import get_embeddings
from langchain_chroma import Chroma

embeddings = get_embeddings()
vectorstore = Chroma.from_documents(
    chunks,
    embedding=embeddings,
    collection_name="learning-semantic-search",
)

results = vectorstore.similarity_search(
    "When can a customer get money back?",
    k=2,
)

for i, doc in enumerate(results, 1):
    print(i, doc.metadata, doc.page_content)

## Exercise: retrieval evaluation before generation

Tạo ít nhất 10 queries với expected relevant source.

Metrics tối thiểu:
- Hit@k
- Recall@k
- MRR (optional but recommended)

Thử:
- chunk size 200 / 500 / 1000;
- k = 2 / 4 / 8;
- metadata filters.

## Required output
Một bảng retrieval benchmark. Không dùng LLM để “che” retrieval kém.

## Done criteria
Bạn có thể chứng minh retrieval configuration tốt hơn bằng dataset/metric, không chỉ bằng cảm giác.